# Modelisation - K-Nearest Neighbors (KNN)

Pipeline complet pour la prediction de la gravite des accidents de la route (2024).

**Etapes :** chargement - nettoyage - encodage - normalisation - entrainement - evaluation - validation croisee - analyse de k.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Chargement et Exploration des Donnees

In [ ]:
base_path = Path.cwd().parent
df = pd.read_csv(base_path / 'data' / 'raw' / 'accidents_2024.csv', sep=';', encoding='latin-1')
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravite (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Donnees chargees : {df.shape[0]} lignes, {df.shape[1]} colonnes')
print(df['Gravite (label)'].value_counts())


## 2. Nettoyage et Preparation des Donnees

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravite (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Lignes apres nettoyage : {df.shape[0]}')
print(df['Gravite (label)'].value_counts())


## 3. Encodage, Decoupage Train/Test et Normalisation

Le KNN etant base sur les distances euclidiennes, la normalisation (StandardScaler) est indispensable pour eviter que les variables a grande echelle dominent le calcul.

In [ ]:
y = df['Gravite (label)']
X_raw = df.drop(columns=['Num_Acc', 'Departement', 'Gravite (label)'])
X_encoded = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train : {X_train_scaled.shape} | Test : {X_test_scaled.shape}')


## 4. Modelisation avec KNeighborsClassifier

### Choix des hyperparametres
- **k=80** : valeur selectionnee apres exploration (voir section 8)
- **weights='distance'** : les voisins proches ont plus de poids, reduisant le bruit

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=80, weights='distance')
model_knn.fit(X_train_scaled, y_train)

y_pred = model_knn.predict(X_test_scaled)
print('Modele entraine.')


## 5. Evaluation du Modele

### Metriques de performance
- **Accuracy** : proportion de predictions correctes
- **Recall** : proportion de vrais positifs detectes *(prioritaire pour la classe Tue)*
- **F1-Score** : moyenne harmonique precision/recall

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred, label=''):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(label)
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    return acc, prec, rec, f1

evaluate_model(y_test, y_pred, 'KNN - Ensemble de test')
print()
print(classification_report(y_test, y_pred))


## 6. Matrice de Confusion

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matrice de confusion - KNN (k=80, weights=distance)', fontweight='bold')
ax.set_xlabel('Predit')
ax.set_ylabel('Reel')
plt.tight_layout()
plt.show()


## 7. Validation Croisee (5-Fold Stratifie)

La validation croisee stratifiee garantit la representation de chaque classe dans chaque pli.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_knn, X_train_scaled, y_train,
                          cv=cv, scoring='recall_macro')

print(f'Recall macro - CV 5-fold : {scores.mean():.3f} (+/- {scores.std():.3f})')
print(f'Scores par fold : {scores.round(3)}')


## 8. Analyse de l'Influence du Parametre k

In [ ]:
k_values = [5, 10, 20, 40, 60, 80, 100]
recalls = []
for k in k_values:
    knn_tmp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    sc = cross_val_score(knn_tmp, X_train_scaled, y_train,
                         cv=StratifiedKFold(3, shuffle=True, random_state=42),
                         scoring='recall_macro')
    recalls.append(sc.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, recalls, marker='o', color='steelblue')
ax.axvline(x=80, color='red', linestyle='--', label='k=80 retenu')
ax.set_title('Recall macro selon la valeur de k', fontweight='bold')
ax.set_xlabel('k (nombre de voisins)')
ax.set_ylabel('Recall macro (CV 3-fold)')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Resume et Recommandations

In [ ]:
print('=' * 65)
print('RESUME - KNN')
print('=' * 65)
print('  n_neighbors : 80 | weights : distance')
print('  Normalisation : StandardScaler (obligatoire)')
print(f'  Recall macro CV : {scores.mean():.3f}')
print()
print('  Avantages : simple, non parametrique, pas hypothese sur la distribution.')
print('  Limites   : lent en prediction, sensible aux dimensions elevees.')
